# Public/private key pairs  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.4.a: RSA key-pair parameters

**Skill:** Mitigate Risk · **Maps to:** 5.4.A · **Language:** Python 3 (pytest)

## Problem
Before RSA can encrypt or sign anything, it needs a key pair. Given two prime numbers p
and q, implement the first steps of RSA key generation described in `atom.md`: compute
the modulus n, compute phi(n), and validate whether a proposed public/private exponent
pair (e, d) actually satisfies the required mathematical relationship.

## Function contract
Implement, in the code editor:
- `rsa_params(p: int, q: int) -> tuple`, given two (small, distinct) primes, return
  `(n, phi_n)` where `n = p * q` and `phi_n = (p - 1) * (q - 1)`.
- `is_valid_keypair(e: int, d: int, phi_n: int) -> bool`, return `True` iff
  `(e * d) % phi_n == 1`, i.e. d is the modular inverse of e mod phi_n.
- `solve(data: dict) -> dict`, the blackbox entry point that dispatches to the two
  functions above (see the Reference section below).

## The math (see atom.md)
```
n       = p * q
phi(n)  = (p - 1) * (q - 1)
valid keypair (e, d)  <=>  (e * d) mod phi(n) == 1
```
The public key is `(e, n)`; the private key is `(d, n)`. Both share `n`, but `d` cannot
be recovered from `e` and `n` alone without knowing `phi(n)` (which requires factoring
`n` back into `p` and `q`).

## Input / Output
- `rsa_params(p, q)`: `p, q: int` (primes) -> `(n, phi_n)` tuple of `int`.
- `is_valid_keypair(e, d, phi_n)`: all `int` -> `bool`.
- `solve(data)`: `data = {"op": "params", "p": <int>, "q": <int>}` ->
  `{"n": <int>, "phi_n": <int>}`, or
  `data = {"op": "validate", "e": <int>, "d": <int>, "phi_n": <int>}` ->
  `{"valid": <bool>}`.

## Examples
```
rsa_params(3, 11)  -> (33, 20)     # n = 3*11 = 33, phi(n) = 2*10 = 20
rsa_params(5, 11)  -> (55, 40)     # n = 5*11 = 55, phi(n) = 4*10 = 40

is_valid_keypair(e=3, d=7, phi_n=20)  -> True   # 3*7 = 21, 21 % 20 == 1
is_valid_keypair(e=3, d=6, phi_n=20)  -> False  # 3*6 = 18, 18 % 20 != 1
```

## Constraints
- Deterministic, offline, standard library only. No external crypto/bignum libraries, this problem implements the underlying arithmetic directly.
- Inputs are small, textbook-scale integers as in the atom's worked examples; no
  primality checking is required.
- Must pass all cases in the tests (visible + hidden), including the p=3,q=11 and
  p=5,q=11 examples from `atom.md`, and both valid and invalid keypair checks.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry points
```python
def rsa_params(p: int, q: int) -> tuple
def is_valid_keypair(e: int, d: int, phi_n: int) -> bool
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; `rsa_params()` and `is_valid_keypair()` are the
underlying primitives it's built on and are tested directly as well.

### Input
- `rsa_params(p, q)`: `p` and `q` are (small, distinct) prime integers.
- `is_valid_keypair(e, d, phi_n)`: `e`, `d`, `phi_n` are all `int`.
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"params"` or `"validate"`.
  - `"params"`: `{"op": "params", "p": <int>, "q": <int>}`
  - `"validate"`: `{"op": "validate", "e": <int>, "d": <int>, "phi_n": <int>}`

### Output (`result`)
- `rsa_params(p, q)` returns a 2-tuple `(n, phi_n)` where `n = p * q` and
  `phi_n = (p - 1) * (q - 1)`.
- `is_valid_keypair(e, d, phi_n)` returns `bool`: `True` iff `(e * d) % phi_n == 1`.
- `solve(...)` returns `{"n": <int>, "phi_n": <int>}` for op `"params"`, or
  `{"valid": <bool>}` for op `"validate"`.

### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only, no external
  crypto/bignum libraries; this is the from-scratch pedagogical RSA-parameter math
  described in atom.md (n = p*q, phi(n) = (p-1)(q-1), e*d ≡ 1 mod phi(n)).
- Inputs are small, textbook-scale integers (not cryptographically secure primes); no
  primality checking is required inside `rsa_params`, callers are trusted to pass primes.
- `is_valid_keypair` must work for any nonnegative `e`, `d`, `phi_n` combination, including
  cases where the pair is deliberately invalid (used to test rejection).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.4.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests).

Implements the RSA key-pair math by hand (n, phi(n), keypair validation) as described
in atom.md, using small textbook-scale primes. No external crypto libraries — this is
a from-scratch pedagogical implementation: crypto problems
implement the underlying math/logic directly.
"""


def rsa_params(p: int, q: int) -> tuple:
    """Given two (small, distinct) primes p and q, return (n, phi_n) where
    n = p * q and phi_n = (p - 1) * (q - 1).

    This is step 1-2 of RSA key generation (see atom.md): n becomes part of both
    the public and private key; phi_n is used to derive the private exponent d.
    """
    n = p * q
    phi_n = (p - 1) * (q - 1)
    return (n, phi_n)


def is_valid_keypair(e: int, d: int, phi_n: int) -> bool:
    """Return True iff (e * d) mod phi_n == 1, i.e. d is the modular inverse of e
    mod phi_n. This is the defining relationship between the public exponent e
    and the private exponent d (see atom.md step 4)."""
    return (e * d) % phi_n == 1


def solve(data: dict) -> dict:
    """Blackbox entry point. Dispatches on data['op']:
      - {"op": "params", "p": <int>, "q": <int>}
            -> {"n": <int>, "phi_n": <int>}
      - {"op": "validate", "e": <int>, "d": <int>, "phi_n": <int>}
            -> {"valid": <bool>}
    """
    op = data["op"]
    if op == "params":
        n, phi_n = rsa_params(data["p"], data["q"])
        return {"n": n, "phi_n": phi_n}
    elif op == "validate":
        return {"valid": is_valid_keypair(data["e"], data["d"], data["phi_n"])}
    else:
        raise ValueError(f"unknown op: {op!r}")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _oracle_params(p, q):
    """Independent reference computation of (n, phi_n), used only to check the
    student implementation's output, never as part of the implementation."""
    return (p * q, (p - 1) * (q - 1))


# ---- visible cases -------------------------------------------------------
def test_rsa_params_small_primes_3_11():
    # worked example from atom.md / quiz.yaml q3
    n, phi_n = impl.rsa_params(3, 11)
    assert (n, phi_n) == (33, 20)


def test_rsa_params_small_primes_5_11():
    # worked example from atom.md's "Try it" section
    n, phi_n = impl.rsa_params(5, 11)
    assert (n, phi_n) == (55, 40)


def test_rsa_params_matches_oracle():
    p, q = 7, 13
    got = impl.rsa_params(p, q)
    assert got == _oracle_params(p, q)


def test_is_valid_keypair_true_case():
    # e=3, d=7, phi_n=20: 3*7 = 21, 21 % 20 == 1
    assert impl.is_valid_keypair(3, 7, 20) is True


def test_is_valid_keypair_false_case():
    # e=3, d=6, phi_n=20: 3*6 = 18, 18 % 20 != 1
    assert impl.is_valid_keypair(3, 6, 20) is False


def test_solve_params_op():
    result = impl.solve({"op": "params", "p": 3, "q": 11})
    assert result == {"n": 33, "phi_n": 20}


def test_solve_validate_op_valid():
    result = impl.solve({"op": "validate", "e": 3, "d": 7, "phi_n": 20})
    assert result == {"valid": True}


# ---- hidden / edge cases -------------------------------------------------
def test_rsa_params_return_type_is_tuple_like():
    n, phi_n = impl.rsa_params(3, 11)  # unpacking must work
    assert n == 33 and phi_n == 20


def test_solve_validate_op_invalid():
    result = impl.solve({"op": "validate", "e": 3, "d": 6, "phi_n": 20})
    assert result == {"valid": False}


def test_is_valid_keypair_e_d_larger_than_phi_n():
    # e=23, d=7, phi_n=20: 23*7 = 161, 161 % 20 == 1 -> valid even though e > phi_n
    assert impl.is_valid_keypair(23, 7, 20) is True


def test_is_valid_keypair_off_by_one_is_invalid():
    # e=3, d=8, phi_n=20: 3*8 = 24, 24 % 20 == 4, not 1
    assert impl.is_valid_keypair(3, 8, 20) is False


def test_rsa_params_larger_primes():
    p, q = 17, 23
    n, phi_n = impl.rsa_params(p, q)
    assert (n, phi_n) == _oracle_params(p, q)


def test_full_keygen_roundtrip_p5_q11_e3():
    # From p=5, q=11: n=55, phi_n=40. e=3 has inverse d=27 mod 40 (3*27=81, 81%40=1).
    n, phi_n = impl.rsa_params(5, 11)
    assert (n, phi_n) == (55, 40)
    assert impl.is_valid_keypair(3, 27, phi_n) is True
    assert impl.is_valid_keypair(3, 13, phi_n) is False


def test_solve_params_op_matches_oracle_random_primes():
    p, q = 13, 19
    result = impl.solve({"op": "params", "p": p, "q": q})
    expected_n, expected_phi = _oracle_params(p, q)
    assert result == {"n": expected_n, "phi_n": expected_phi}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
